# 043. Small comparison: 8B vs 70B generation quality

Цель ноутбука — на маленькой выборке сравнить качество ответов двух LLM-моделей: более дешёвой/быстрой 8B и более сильной 70B.

Проверка специально сделана небольшой, чтобы не тратить много лимитов. По умолчанию используется **один и тот же already saved controlled-контекст** из `03a`, чтобы изолировать именно влияние модели генерации, а не retrieval.

Что сравниваем:
- долю `I don't know`;
- EM/F1 относительно gold answer;
- среднюю длину ответа;
- парные примеры: вопрос, gold, ответ 8B, ответ 70B.

Если нужно, режим можно переключить на другую выборку/папку, но для первичной проверки лучше оставить маленький размер.

In [8]:
# Если запускаете в Colab, сначала подключите Google Drive
from google.colab import drive
try:
    drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped or failed:', e)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
# Установка зависимостей при необходимости
# Раскомментируйте, если библиотек нет в окружении.
!pip -q install groq pandas tqdm


In [10]:
import os
import re
import json
import time
import random
from pathlib import Path
from typing import Any, Dict, List

import pandas as pd
from tqdm.auto import tqdm

try:
    from groq import Groq
except Exception as e:
    Groq = None
    print('Groq is not installed. Run: !pip install groq')


## 1. Настройки

По умолчанию берём controlled-ответы из `03a`, потому что там уже сохранены вопросы, gold answers и контексты. Это позволяет сравнить **только силу модели генерации**: обе модели получают один и тот же вопрос и один и тот же контекст.

In [11]:
# ====== PATHS ======
ARTIFACTS_DIR = Path('/content/drive/MyDrive/rag_experiment/artifacts')
CONTROLLED_ANSWERS_DIR = ARTIFACTS_DIR / 'answers' / 'controlled'
OUT_DIR = ARTIFACTS_DIR / 'model_comparison_8b_vs_70b'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ====== SAMPLE SETTINGS ======
# Для первого запуска лучше оставить baseline и 5 примеров на тип шума.
TEST_METHODS = ['baseline']
NOISE_TYPES = ['semantic_distractors', 'counterfactuals', 'structural']
NOISE_LEVELS = [80]
SAMPLE_PER_CONFIG = 5
RANDOM_SEED = 42

# Если хочется проверить не только самый тяжёлый шум, можно поставить [40, 80].
# NOISE_LEVELS = [40, 80]

# ====== MODELS ======
# Проверьте актуальные названия моделей в вашем Groq-аккаунте.
MODEL_8B = 'llama-3.1-8b-instant'
MODEL_70B = 'llama-3.3-70b-versatile'

# ====== GENERATION SETTINGS ======
TEMPERATURE = 0.0
MAX_TOKENS = 96
REQUEST_SLEEP_SECONDS = 0.3

# ====== CACHE ======
CACHE_PATH = OUT_DIR / 'generations_8b_vs_70b_cache.json'
SUMMARY_CSV = OUT_DIR / 'summary_8b_vs_70b.csv'
DETAILS_CSV = OUT_DIR / 'details_8b_vs_70b.csv'
PAIRWISE_CSV = OUT_DIR / 'pairwise_8b_vs_70b.csv'

print('Controlled dir exists:', CONTROLLED_ANSWERS_DIR.exists())
print('Output dir:', OUT_DIR)


Controlled dir exists: True
Output dir: /content/drive/MyDrive/rag_experiment/artifacts/model_comparison_8b_vs_70b


## 2. Промпт

Используем сбалансированный вариант промпта: модель не должна использовать внешние знания, но должна пытаться ответить, если в контексте есть хотя бы частично релевантные фрагменты.

In [12]:
BALANCED_CONTEXTUAL_PROMPT = """Answer the question using only the provided context.

The context may be noisy, irrelevant, duplicated, corrupted, or partially contradictory.
Your task is to identify the most relevant evidence and provide the most likely short answer based only on that evidence.

Rules:
1. Do not use outside knowledge.
2. If there is any relevant evidence in the context, provide the best possible answer.
3. If several fragments conflict, prefer the fragment that is most directly related to the question.
4. Answer "I don't know" only if the context contains no relevant information at all.
5. Keep the answer concise.

Question:
{question}

Context:
{context}

Short answer:
"""


In [13]:
def normalize_text(s: Any) -> str:
    if s is None:
        return ''
    s = str(s).lower()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = " ".join(s.split())
    return s


def exact_match(pred: str, gold: str) -> int:
    return int(normalize_text(pred) == normalize_text(gold))


def f1_score(pred: str, gold: str) -> float:
    pred_tokens = normalize_text(pred).split()
    gold_tokens = normalize_text(gold).split()
    if len(pred_tokens) == 0 and len(gold_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0
    common = {}
    for t in pred_tokens:
        common[t] = common.get(t, 0) + 1
    overlap = 0
    for t in gold_tokens:
        if common.get(t, 0) > 0:
            overlap += 1
            common[t] -= 1
    if overlap == 0:
        return 0.0
    precision = overlap / len(pred_tokens)
    recall = overlap / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

IDK_PATTERNS = [
    r"\bi don'?t know\b",
    r"\bunknown\b",
    r"\bnot enough information\b",
    r"\bcannot answer\b",
    r"\bcan't answer\b",
    r"\binsufficient information\b",
    r"\bno relevant information\b",
]

def is_idk(answer: str) -> bool:
    a = str(answer or '').strip().lower()
    return any(re.search(p, a) for p in IDK_PATTERNS)


def load_json(path: Path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    tmp.replace(path)


## 3. Извлечение контекста из сохранённых controlled-файлов

Код ниже старается быть устойчивым к разным форматам JSON: словарь `qid -> record` или список записей. Контекст ищется в полях `final_context`, `input_context`, `context_docs`, `context`, `docs`, `retrieved_docs`.

In [14]:
def records_from_loaded_json(data: Any) -> List[Dict[str, Any]]:
    if isinstance(data, list):
        return [x for x in data if isinstance(x, dict)]
    if isinstance(data, dict):
        records = []
        for k, v in data.items():
            if isinstance(v, dict):
                rec = dict(v)
                rec.setdefault('qid', k)
                records.append(rec)
        return records
    return []


def get_question(rec: Dict[str, Any]) -> str:
    return rec.get('question') or rec.get('query') or rec.get('q') or ''


def get_gold(rec: Dict[str, Any]) -> str:
    gold = rec.get('gold') or rec.get('gold_answer') or rec.get('answer') or rec.get('reference') or rec.get('target')
    if isinstance(gold, list):
        return str(gold[0]) if gold else ''
    return str(gold or '')


def get_context_docs(rec: Dict[str, Any]) -> List[Any]:
    for key in ['final_context', 'input_context', 'context_docs', 'context', 'docs', 'retrieved_docs']:
        val = rec.get(key)
        if isinstance(val, list) and val:
            return val
        if isinstance(val, str) and val.strip():
            return [val]
    return []


def format_context(docs: List[Any], max_chars_per_doc: int = 1200, max_docs: int = 8) -> str:
    parts = []
    for i, d in enumerate(docs[:max_docs], 1):
        if isinstance(d, str):
            title, role, text = '', '', d
        elif isinstance(d, dict):
            title = d.get('title', '') or d.get('doc_title', '')
            role = d.get('role', '')
            text = d.get('text', '') or d.get('content', '') or d.get('page_content', '') or ''
        else:
            title, role, text = '', '', str(d)
        text = str(text)[:max_chars_per_doc]
        header = f'[{i}]'
        if title:
            header += f' Title: {title}'
        if role:
            header += f' | Role: {role}'
            parts.append(f'{header}{text}')
    return ''.join(parts)


def build_items() -> List[Dict[str, Any]]:
    random.seed(RANDOM_SEED)
    items = []
    for method in TEST_METHODS:
        for noise_type in NOISE_TYPES:
            for noise_level in NOISE_LEVELS:
                path = CONTROLLED_ANSWERS_DIR / f'{method}__{noise_type}__lvl{noise_level}.json'
                if not path.exists():
                    print('Missing:', path)
                    continue
                data = load_json(path)
                records = records_from_loaded_json(data)
                usable = []
                for rec in records:
                    q = get_question(rec)
                    gold = get_gold(rec)
                    docs = get_context_docs(rec)
                    ctx = format_context(docs)
                    if q and gold and ctx:
                        usable.append({
                            'qid': rec.get('qid', ''),
                            'method': method,
                            'noise_type': noise_type,
                            'noise_level': noise_level,
                            'question': q,
                            'gold': gold,
                            'context': ctx,
                        })
                if not usable:
                    print('No usable records in:', path)
                    if records:
                        print('Example keys:', sorted(records[0].keys()))
                    continue
                random.shuffle(usable)
                items.extend(usable[:SAMPLE_PER_CONFIG])
                print(f'{method}/{noise_type}/lvl{noise_level}: usable={len(usable)}, sampled={min(SAMPLE_PER_CONFIG, len(usable))}')
    return items

items = build_items()
print('Total test items:', len(items))


baseline/semantic_distractors/lvl80: usable=100, sampled=5
baseline/counterfactuals/lvl80: usable=100, sampled=5
baseline/structural/lvl80: usable=100, sampled=5
Total test items: 15


## 4. Groq generation с кэшем

Каждый пример генерируется двумя моделями. Результаты сохраняются в кэш, поэтому при обрыве/лимитах можно перезапустить ноутбук и продолжить.

In [15]:
if Groq is None:
    raise RuntimeError('Groq is not installed. Run: !pip install groq')

if not os.environ.get('GROQ_API_KEY'):
    try:
        from getpass import getpass
        os.environ['GROQ_API_KEY'] = getpass('Enter GROQ_API_KEY: ')
    except Exception:
        pass

client = Groq(api_key=os.environ.get('GROQ_API_KEY'))


def call_groq(model: str, prompt: str, max_retries: int = 3) -> str:
    last_err = None
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[{'role': 'user', 'content': prompt}],
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS,
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            last_err = e
            msg = str(e).lower()
            if any(x in msg for x in ['rate limit', 'quota', 'tokens per minute', 'too many requests', '429']):
                raise RuntimeError(f'LIMIT_REACHED: {e}')
            time.sleep(1.5 * (attempt + 1))
    raise RuntimeError(f'Groq call failed: {last_err}')

cache = load_json(CACHE_PATH) if CACHE_PATH.exists() else {}
print('Cached generations:', len(cache))


Enter GROQ_API_KEY: ··········
Cached generations: 0


In [17]:
def item_key(item: Dict[str, Any], model_label: str) -> str:
    return '|'.join([
        str(item.get('method', '')),
        str(item.get('noise_type', '')),
        str(item.get('noise_level', '')),
        str(item.get('qid', '')),
        model_label,
    ])

models = {
    '8b': MODEL_8B,
    '70b': MODEL_70B,
}

try:
    for item in tqdm(items, desc='8B vs 70B prompt test'):
        prompt = BALANCED_CONTEXTUAL_PROMPT.format(question=item['question'], context=item['context'])
        for label, model_name in models.items():
            key = item_key(item, label)
            if key in cache:
                continue
            answer = call_groq(model_name, prompt)
            cache[key] = {
                **{k: item[k] for k in ['qid', 'method', 'noise_type', 'noise_level', 'question', 'gold']},
                'model_label': label,
                'model_name': model_name,
                'answer': answer,
            }
            save_json(cache, CACHE_PATH)
            time.sleep(REQUEST_SLEEP_SECONDS)
except RuntimeError as e:
    save_json(cache, CACHE_PATH)
    print('Stopped safely:', e)
    print('Cache saved to:', CACHE_PATH)

print('Cached generations:', len(cache))


8B vs 70B prompt test:   0%|          | 0/15 [00:00<?, ?it/s]

Cached generations: 30


## 5. Подсчёт метрик

In [18]:
rows = []
for rec in cache.values():
    pred = rec.get('answer', '')
    gold = rec.get('gold', '')
    row = dict(rec)
    row['EM'] = exact_match(pred, gold)
    row['F1'] = f1_score(pred, gold)
    row['is_idk'] = int(is_idk(pred))
    row['answer_len_words'] = len(str(pred).split())
    rows.append(row)

df = pd.DataFrame(rows)
if df.empty:
    print('No generations yet.')
else:
    display(df.head())
    df.to_csv(DETAILS_CSV, index=False)
    print('Saved details:', DETAILS_CSV)


,qid,method,noise_type,noise_level,question,gold,model_label,model_name,answer,EM,F1,is_idk,answer_len_words
0,5ab52cb2554299637185c4f1,baseline,semantic_distractors,80,In what city did Nancy Winstel coach women's c...,Highland Heights,8b,llama-3.1-8b-instant,I don't know.,0,0.000000,1,3
1,5ab52cb2554299637185c4f1,baseline,semantic_distractors,80,In what city did Nancy Winstel coach women's c...,Highland Heights,70b,llama-3.3-70b-versatile,"Highland Heights, Kentucky (implied as Norther...",0,0.235294,0,15
2,5a7a68d25542996c55b2dd98,baseline,semantic_distractors,80,How is the namesake of The Mountbatten Institu...,second cousin once removed,8b,llama-3.1-8b-instant,I don't know.,0,0.000000,1,3
3,5a7a68d25542996c55b2dd98,baseline,semantic_distractors,80,How is the namesake of The Mountbatten Institu...,second cousin once removed,70b,llama-3.3-70b-versatile,"Louis Mountbatten, the namesake of The Mountba...",0,0.153846,0,53
4,5a78f753554299078472776e,baseline,semantic_distractors,80,"Which musician is older, Philip Labonte or Ale...",Philip Steven Labonte,8b,llama-3.1-8b-instant,"Alexi Laiho was born in 1979, but the exact da...",0,0.071429,0,56


Saved details: /content/drive/MyDrive/rag_experiment/artifacts/model_comparison_8b_vs_70b/details_8b_vs_70b.csv


In [19]:
if not df.empty:
    summary = (
        df.groupby(['model_label', 'model_name', 'noise_type', 'noise_level'], dropna=False)
          .agg(
              n=('answer', 'count'),
              EM=('EM', 'mean'),
              F1=('F1', 'mean'),
              idk_count=('is_idk', 'sum'),
              idk_rate=('is_idk', 'mean'),
              avg_answer_len_words=('answer_len_words', 'mean'),
          )
          .reset_index()
          .sort_values(['noise_type', 'noise_level', 'model_label'])
    )
    display(summary)
    summary.to_csv(SUMMARY_CSV, index=False)
    print('Saved summary:', SUMMARY_CSV)


,model_label,model_name,noise_type,noise_level,n,EM,F1,idk_count,idk_rate,avg_answer_len_words
0,70b,llama-3.3-70b-versatile,counterfactuals,80,5,0.2,0.266667,0,0.0,5.2
3,8b,llama-3.1-8b-instant,counterfactuals,80,5,0.0,0.114815,2,0.4,8.8
1,70b,llama-3.3-70b-versatile,semantic_distractors,80,5,0.0,0.121768,0,0.0,35.6
4,8b,llama-3.1-8b-instant,semantic_distractors,80,5,0.0,0.014286,3,0.6,16.8
2,70b,llama-3.3-70b-versatile,structural,80,5,0.4,0.754872,0,0.0,4.6
5,8b,llama-3.1-8b-instant,structural,80,5,0.2,0.358333,3,0.6,5.0


Saved summary: /content/drive/MyDrive/rag_experiment/artifacts/model_comparison_8b_vs_70b/summary_8b_vs_70b.csv


## 6. Парное сравнение ответов

Эта таблица удобна для ручного просмотра: один и тот же вопрос, gold answer, ответ 8B и ответ 70B.

In [20]:
if not df.empty:
    key_cols = ['method', 'noise_type', 'noise_level', 'qid', 'question', 'gold']
    wide = df.pivot_table(
        index=key_cols,
        columns='model_label',
        values='answer',
        aggfunc='first'
    ).reset_index()
    wide.columns.name = None

    metric_wide = df.pivot_table(
        index=key_cols,
        columns='model_label',
        values=['EM', 'F1', 'is_idk'],
        aggfunc='first'
    ).reset_index()
    metric_wide.columns = [('_'.join([str(x) for x in col if x]) if isinstance(col, tuple) else col) for col in metric_wide.columns]

    pairwise = wide.merge(metric_wide, on=key_cols, how='left')
    pairwise.to_csv(PAIRWISE_CSV, index=False)
    display(pairwise.head(20))
    print('Saved pairwise:', PAIRWISE_CSV)


,method,noise_type,noise_level,qid,question,gold,70b,8b,EM_70b,EM_8b,F1_70b,F1_8b,is_idk_70b,is_idk_8b
0,baseline,counterfactuals,80,5a735bae55429901807dafef,Which indigenous people of the Ryukyu Islands ...,Ryukyuan people,Ryūkyūan (also referred to as Ainu in one of t...,The Ainu people.,0,0,0.000000,0.500000,0,0
1,baseline,counterfactuals,80,5a8e463a5542995a26add4b7,Havelock Ellis and Arnold Bennett were both what?,writer,English writers.,English writers.,0,0,0.000000,0.000000,0,0
2,baseline,counterfactuals,80,5ab1d7ac554299449642c7e6,"Which genus has more species, Xanthoceras or E...",Ehretia,Ehretia.,I don't know. \n\nThe context mentions that Eh...,1,0,1.000000,0.074074,0,1
3,baseline,counterfactuals,80,5ac434ed554299204fd21f09,"""Coming Over"" is the second Japanese single a...",2012,2011,I don't know.,0,0,0.000000,0.000000,0,1
4,baseline,counterfactuals,80,5ade15f35542997545bbbe47,Katherine Waterston and Chrisann Brennan has w...,Steve Jobs,Katherine Waterston portrayed Chrisann Brennan...,They both portrayed Chrisann Brennan in a film.,0,0,0.333333,0.000000,0,0
5,baseline,semantic_distractors,80,5a7268235542992359bc3081,What unit of the Air National Guard is located...,133d Air Refueling Squadron,Pease Air National Guard Base was home to the ...,I don't know.,0,0,0.059701,0.000000,0,1
6,baseline,semantic_distractors,80,5a78f753554299078472776e,"Which musician is older, Philip Labonte or Ale...",Philip Steven Labonte,Philip Labonte is older. He was born on April ...,"Alexi Laiho was born in 1979, but the exact da...",0,0,0.160000,0.071429,0,0
7,baseline,semantic_distractors,80,5a7a68d25542996c55b2dd98,How is the namesake of The Mountbatten Institu...,second cousin once removed,"Louis Mountbatten, the namesake of The Mountba...",I don't know.,0,0,0.153846,0.000000,0,1
8,baseline,semantic_distractors,80,5a8b3c5b55429971feec4684,"Are Milk Lake, Taiwan and Larnaca Salt Lake bo...",no,"Yes, Milk Lake is in Taiwan, and Larnaca Salt ...","Milk Lake, Taiwan is located in Asia. \n\nHowe...",0,0,0.000000,0.000000,0,0
9,baseline,semantic_distractors,80,5ab52cb2554299637185c4f1,In what city did Nancy Winstel coach women's c...,Highland Heights,"Highland Heights, Kentucky (implied as Norther...",I don't know.,0,0,0.235294,0.000000,0,1


Saved pairwise: /content/drive/MyDrive/rag_experiment/artifacts/model_comparison_8b_vs_70b/pairwise_8b_vs_70b.csv


## 7. Как интерпретировать

Для первичного решения “стоит ли мучиться с лимитами 70B” смотрим:

1. Насколько 70B уменьшает `I don't know` относительно 8B.
2. Насколько растёт F1/EM на одинаковых вопросах и контекстах.
3. Есть ли качественная разница в ручном просмотре `pairwise` таблицы.
4. Не становится ли 70B более многословной или более склонной выбирать шумовой фрагмент.

Если прирост маленький, для массовых прогонов можно оставить 8B, а 70B использовать только как judge или для финальной перепроверки спорных конфигураций.